# 05 — Simulación Monte Carlo paralela

## Objetivo

Distribuir la ejecución de escenarios Monte Carlo entre múltiples
procesos utilizando ProcessPoolExecutor.

Cada proceso recibe un bloque independiente de escenarios.

La implementación se diseña para funcionar correctamente en
Windows mediante funciones definidas a nivel de módulo.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import numpy as np
import pandas as pd

In [3]:
from src.data.loader import load_csv

from src.simulation.model import (
    NormalReturnModel,
)

from src.simulation.sequential import (
    SimulationConfig,
)

from src.simulation.parallel import (
    simulate_parallel_final_values,
)

In [4]:
PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "sp500_returns.csv"
)

returns_data = load_csv(
    PROCESSED_DATA_PATH
)

mean_return = (
    returns_data["log_return"].mean()
)

volatility = (
    returns_data["log_return"].std()
)

model = NormalReturnModel(
    mean=mean_return,
    volatility=volatility,
)

In [5]:
config = SimulationConfig(
    scenarios=1_000,
    horizon=252,
    initial_value=100.0,
    seed=42,
)

print(config)

SimulationConfig(scenarios=1000, horizon=252, initial_value=100.0, seed=42)


In [6]:
workers = min(
    2,
    os.cpu_count() or 1,
)

print(
    f"Workers utilizados: {workers}"
)

Workers utilizados: 2


In [7]:
parallel_values = (
    simulate_parallel_final_values(
        model=model,
        config=config,
        workers=workers,
    )
)

print(
    "Cantidad de resultados:",
    len(parallel_values),
)

Cantidad de resultados: 1000


In [8]:
print(
    parallel_values[:10]
)

[104.95344043 121.20336011 138.28608758 103.6516786   86.49740214
 117.1365523  143.63409566 131.73466767 117.52188982 105.62680581]


In [9]:
parallel_returns = (
    parallel_values - config.initial_value
) / config.initial_value

print(
    f"Rendimiento promedio: "
    f"{parallel_returns.mean():.6f}"
)

print(
    f"Mínimo: "
    f"{parallel_returns.min():.6f}"
)

print(
    f"Máximo: "
    f"{parallel_returns.max():.6f}"
)

Rendimiento promedio: 0.117044
Mínimo: -0.463669
Máximo: 0.936718


## Verificación

La implementación paralela debe producir la cantidad esperada de
resultados y conservar la misma estructura conceptual que la
simulación secuencial.

Los resultados individuales no tienen que coincidir exactamente
con la implementación secuencial debido a que los generadores
aleatorios utilizan secuencias independientes.